In [0]:
# Task 2.1 & Task 2.2: Cleansing, Type Casting & Daily Deduplication
spark.sql("""
CREATE OR REPLACE TABLE youtube_lakehouse.silver.fact_video_daily_snapshots AS
WITH exploded_bronze AS (
    SELECT 
        _source_batch_id,
        _ingest_timestamp,
        EXPLODE(records) AS payload
    FROM youtube_lakehouse.bronze.raw_videos
),
parsed_fields AS (
    SELECT 
        payload.id AS video_id,
        payload.snippet.channelId AS channel_id,
        payload.snippet.channelTitle AS channel_title,
        payload.snippet.title AS video_title,
        payload.snippet.tags AS raw_tags,
        TO_TIMESTAMP(payload.snippet.publishedAt) AS published_at,
        CAST(payload.statistics.viewCount AS BIGINT) AS cumulative_views,
        CAST(COALESCE(payload.statistics.likeCount, '0') AS BIGINT) AS cumulative_likes,
        CAST(COALESCE(payload.statistics.commentCount, '0') AS BIGINT) AS cumulative_comments,
        CAST(_ingest_timestamp AS DATE) AS snapshot_date,
        _ingest_timestamp,
        ROW_NUMBER() OVER (
            PARTITION BY payload.id, CAST(_ingest_timestamp AS DATE) 
            ORDER BY _ingest_timestamp DESC
        ) AS intraday_rank
    FROM exploded_bronze
    WHERE payload.id IS NOT NULL
),
daily_deduped AS (
    SELECT * 
    FROM parsed_fields 
    WHERE intraday_rank = 1
)
SELECT 
    video_id,
    channel_id,
    channel_title,
    video_title,
    raw_tags,
    published_at,
    snapshot_date,
    DATEDIFF(snapshot_date, CAST(published_at AS DATE)) AS days_since_published,
    cumulative_views,
    cumulative_likes,
    cumulative_comments,
    _ingest_timestamp
FROM daily_deduped;
""")

print("Table `youtube_lakehouse.silver.fact_video_daily_snapshots` created.")

In [0]:
# Task 2.3: Compute 24-Hour Velocity & Engagement Metrics
spark.sql("""
CREATE OR REPLACE TABLE youtube_lakehouse.silver.fact_video_daily_snapshots AS
WITH base_snapshots AS (
    SELECT * FROM youtube_lakehouse.silver.fact_video_daily_snapshots
),
calculated_deltas AS (
    SELECT 
        video_id,
        channel_id,
        channel_title,
        video_title,
        raw_tags,
        published_at,
        snapshot_date,
        days_since_published,
        cumulative_views,
        -- Day-over-day view increase (LAG window across daily snapshots)
        COALESCE(
            cumulative_views - LAG(cumulative_views, 1) OVER (
                PARTITION BY video_id 
                ORDER BY snapshot_date ASC
            ),
            0
        ) AS delta_views_24h,
        cumulative_likes,
        COALESCE(
            cumulative_likes - LAG(cumulative_likes, 1) OVER (
                PARTITION BY video_id 
                ORDER BY snapshot_date ASC
            ),
            0
        ) AS delta_likes_24h,
        cumulative_comments,
        COALESCE(
            cumulative_comments - LAG(cumulative_comments, 1) OVER (
                PARTITION BY video_id 
                ORDER BY snapshot_date ASC
            ),
            0
        ) AS delta_comments_24h,
        -- Engagement rate: (Likes + Comments) / Views * 100
        ROUND(
            COALESCE(
                (cumulative_likes + cumulative_comments) / NULLIF(cumulative_views, 0) * 100, 
                0.0
            ), 
            2
        ) AS engagement_rate_pct,
        _ingest_timestamp
    FROM base_snapshots
)
SELECT * FROM calculated_deltas;
""")

print("Velocity and engagement metrics computed successfully.")

In [0]:
%sql
CREATE OR REPLACE TABLE youtube_lakehouse.silver.fact_video_daily_snapshots AS
WITH base_snapshots AS (
    SELECT 
        payload.id AS video_id,
        payload.snippet.channelId AS channel_id,
        payload.snippet.channelTitle AS channel_title,
        payload.snippet.title AS video_title,
        payload.snippet.tags AS raw_tags,
        TO_TIMESTAMP(payload.snippet.publishedAt) AS published_at,
        CAST(payload.statistics.viewCount AS BIGINT) AS cumulative_views,
        CAST(COALESCE(payload.statistics.likeCount, '0') AS BIGINT) AS cumulative_likes,
        CAST(COALESCE(payload.statistics.commentCount, '0') AS BIGINT) AS cumulative_comments,
        CAST(_ingest_timestamp AS DATE) AS snapshot_date,
        _ingest_timestamp,
        ROW_NUMBER() OVER (
            PARTITION BY payload.id, CAST(_ingest_timestamp AS DATE) 
            ORDER BY _ingest_timestamp DESC
        ) AS intraday_rank
    FROM (
        SELECT EXPLODE(records) AS payload, _ingest_timestamp 
        FROM youtube_lakehouse.bronze.raw_videos
    )
    WHERE payload.id IS NOT NULL
),
deduped_snapshots AS (
    SELECT * FROM base_snapshots WHERE intraday_rank = 1
),
calculated_deltas AS (
    SELECT 
        video_id,
        channel_id,
        channel_title,
        video_title,
        raw_tags,
        published_at,
        snapshot_date,
        DATEDIFF(snapshot_date, CAST(published_at AS DATE)) AS days_since_published,
        cumulative_views,
        -- True day-over-day delta (allowing negative adjustments)
        COALESCE(
            cumulative_views - LAG(cumulative_views, 1) OVER (
                PARTITION BY video_id 
                ORDER BY snapshot_date ASC
            ),
            0
        ) AS delta_views_24h,
        cumulative_likes,
        COALESCE(
            cumulative_likes - LAG(cumulative_likes, 1) OVER (
                PARTITION BY video_id 
                ORDER BY snapshot_date ASC
            ),
            0
        ) AS delta_likes_24h,
        cumulative_comments,
        COALESCE(
            cumulative_comments - LAG(cumulative_comments, 1) OVER (
                PARTITION BY video_id 
                ORDER BY snapshot_date ASC
            ),
            0
        ) AS delta_comments_24h,
        ROUND(
            COALESCE(
                (cumulative_likes + cumulative_comments) / NULLIF(cumulative_views, 0) * 100, 
                0.0
            ), 
            2
        ) AS engagement_rate_pct,
        -- Audit flag to highlight YouTube view adjustment events
        CASE 
            WHEN (cumulative_views - LAG(cumulative_views, 1) OVER (
                PARTITION BY video_id 
                ORDER BY snapshot_date ASC
            )) < 0 THEN TRUE 
            ELSE FALSE 
        END AS is_view_audit_event,
        _ingest_timestamp
    FROM deduped_snapshots
)
SELECT * FROM calculated_deltas;

In [0]:
# Task 2.4: Tokenize, Explode, and Normalize Video Topic Tags
spark.sql("""
CREATE OR REPLACE TABLE youtube_lakehouse.silver.dim_video_tags AS
SELECT 
    video_id,
    LOWER(TRIM(tag_token)) AS normalized_tag
FROM (
    SELECT 
        video_id,
        EXPLODE(raw_tags) AS tag_token
    FROM youtube_lakehouse.silver.fact_video_daily_snapshots
    WHERE raw_tags IS NOT NULL
)
WHERE LENGTH(TRIM(tag_token)) > 1
GROUP BY video_id, LOWER(TRIM(tag_token));
""")

print("Table `youtube_lakehouse.silver.dim_video_tags` materialized.")

In [0]:
%sql
SELECT 
    video_id,
    video_title,
    snapshot_date,
    days_since_published,
    cumulative_views,
    delta_views_24h,
    engagement_rate_pct
FROM youtube_lakehouse.silver.fact_video_daily_snapshots
ORDER BY delta_views_24h desc, video_id, snapshot_date ASC
LIMIT 12;

In [0]:
# Task 2.5: Adjusted Data Quality Audits
checks = {
    "null_primary_keys": """
        SELECT *
        FROM youtube_lakehouse.silver.fact_video_daily_snapshots 
        WHERE video_id IS NULL OR snapshot_date IS NULL
    """,
    "negative_cumulative_totals": """
        SELECT *
        FROM youtube_lakehouse.silver.fact_video_daily_snapshots 
        WHERE cumulative_views < 0 
           OR cumulative_likes < 0 
           OR cumulative_comments < 0
    """,
    "invalid_engagement_rate": """
        SELECT *
        FROM youtube_lakehouse.silver.fact_video_daily_snapshots 
        WHERE engagement_rate_pct < 0 OR engagement_rate_pct > 100
    """,
    "duplicate_snapshots": """
        SELECT video_id, snapshot_date, COUNT(*) 
        FROM youtube_lakehouse.silver.fact_video_daily_snapshots 
        GROUP BY video_id, snapshot_date 
        HAVING COUNT(*) > 1
    """
}

all_passed = True
for check_name, query in checks.items():
    failed_df = spark.sql(query)
    fail_count = failed_df.count()
    if fail_count > 0:
        print(f"❌ FAILED: {check_name} (Found {fail_count} failing rows)")
        display(failed_df.limit(5))
        all_passed = False
    else:
        print(f"✅ PASSED: {check_name}")

assert all_passed, "Data quality checks failed."
print("\nAll Silver quality constraints passed successfully!")

In [0]:
%sql
ALTER TABLE youtube_lakehouse.silver.fact_video_daily_snapshots 
ADD CONSTRAINT valid_cumulative_views CHECK (cumulative_views >= 0);

ALTER TABLE youtube_lakehouse.silver.fact_video_daily_snapshots 
ADD CONSTRAINT valid_cumulative_likes CHECK (cumulative_likes >= 0);

ALTER TABLE youtube_lakehouse.silver.fact_video_daily_snapshots 
ADD CONSTRAINT valid_engagement_rate CHECK (engagement_rate_pct >= 0 AND engagement_rate_pct <= 100);

In [0]:
%sql
SELECT 
    snapshot_date,
    channel_title,
    video_title,
    cumulative_views,
    delta_views_24h,
    is_view_audit_event
FROM youtube_lakehouse.silver.fact_video_daily_snapshots
WHERE is_view_audit_event = TRUE;